## <span style="color:#0b486b">Part 1: MongoDB Data Model</span>

### Task 1.1 Collection Design

**Vehicle Collection**

This collection is to store static metadata about each registered vehicle and its owner.

Document Schema:
{
  "car_plate": "string",
  "owner_name": "string",
  "owner_addr": "string",
  "vehicle_type": "string",
  "registration_date": "ISODate"
 }
 
Sample Document:
{
  "car_plate": "FT 02",
  "owner_name": "Goh Mei Wei",
  "owner_addr": "943 Jalan Bukit Mawar, Kuala Lumpur",
  "vehicle_type": "Coupe",
  "registration_date": "2006-08-22T03:18:00"
}

Indexes:
Field: car_plate, Type: Single-field index, Purpose: It allows fast lookup of records by registered car plate.

Shard Key Strategy:
Chosen Shard Key: car_plate, Type: Hash-based, Rationale: The documents are uniformly distributed across shards to balance load, since car plates are naturally random.

Data Retention Policy: The data will be stored permanently unless the vehicle is deregistered or its ownership changes.

**Camera Collection**

This collection is to store static information about each camera's physical location, position along the highway and speed limit.

Document Schema:
{
  "camera_id": "int",
  "latitude": "float",
  "longitude": "float",
  "position": "float",
  "speed_limit": "float"
}

Sample Document:
{
  "camera_id": 1,
  "latitude": 2.157730731,
  "longitude": 102.6601002,
  "position": 152.5,
  "speed_limit": 110
}

Indexes:
Field: camera_id, Type: Single-field index, Purpose: It allows quick access to camera configurations by the camera's ID.

Shard Key Strategy:
Chosen Shard Key: camera_id, Type: Range-based, Rationale: Since the number of cameras is only three, range key enables efficient queries.

Data Retention Policy: The data will be stored permanently unless the camera configuration changes.

**Violation Collection**

This collection is storing both flagged instantaneous and average speed-based violations detected by the streaming engine. It has a type field to track the violation type of the vehicle.

Document Schema:
{
  "violation_id": "string",
  "car_plate": "string",
  "violation_date": "String",
  "violations": [
    {
      "violation_type": "String",
      "camera_id": "int",
      "timestamp": "ISODate",
      "speed": "double",
      "speed_limit": float,
    },
    {
      "violation_type": "String",
      "camera_id": "int",
      "timestamp": "ISODate",
      "speed": "double",
      "speed_limit": float,
    }
  ]
  
}

Sample Document:
{
  "violation_id": "0ff38c74-7ee6-41cd-bc26-3a6060604a02",
  "car_plate": "YE 6517",
  "violation_date": "2024-01-01",
  "violations": [
    {
      "violation_type": "instantaneous",
      "camera_id": 1,
      "timestamp": "2024-01-01T08:25:25",
      "speed": 126.7,
      "speed_limit": 110.0,
    },
    {
      "violation_type": "instantaneous",
      "camera_id": 2,
      "timestamp": "2024-01-01T08:25:52.211400",
      "speed": 121.0,
      "speed_limit": 110.0,
    },
    {
      "violation_type": "average_speed",
      "camera_id": 2,
      "timestamp": "2024-01-01T08:25:52.211400",
      "speed": 132.3,
      "speed_limit": 90.0,
    }
  ] 
}

Indexes:
Field: car_plate and violation_date, Type: Compound index, Purpose: It prevents duplicate records for the same vehicle on the same day, and also allows fast check to merge or upsert new violations.

Shard Key Strategy:
Chosen Shard Key: car_plate and violation_date, Type: Hash-based, Rationale: The documents are uniformly distributed and support efficient querying by vehicle and date.

Data Retention Policy: The data will be stored for 5 years to comply with road safety audit and legal requirements.

### Task 1.2 Collection Relationship

The Vehicle collection has a one-to-many relationship with the Violation collection. Each Violation documents stores a reference to Vehicle so that it can avoid duplication of static metadata like owner's address, which are unlikely to change frequently. It can also reduce write cost during ingestion. More importantly, referencing supports maintainability, such as if an owner address is updated, it can be changed in one place without cascading updates.

The Camera collection also has a one-to-many relationship with the Violation collection. Relevant camera data such as camera ID and speed limit are embedded directly in the Violation documents to avoid additional lookups during violation detection. While it introduces potential data duplication if camera configurations change, such changes are infrequent and do not retroactively affect past violations. Therefore, the cost of duplication is outweighed by the benefit of streamlined ingestion and simplified querying.

Strong consistency is achieved by ensuring the streaming engine writes violations using transactional upserts. This guarantees that each violation reflect the latest camera configurations and registered vehicles at the time of detection.

By embedding camera metadata and referencing vehicle data, the model allows the system to support high ingest rates while preserving data consistency and minimizing unnecessary duplication. This design is particularly appropriate for a stream-driven enforcement system where write throughput and real-time responsiveness are primary concerns.

### Task 1.3 Discussion

Consistency and Idempotency:
Our model supports idempotent writes by using upsert with a compound key (car_plate and violation_date). This ensures that if a record already exists for the same vehicle on the same day, new violation entries can be merged into the existing violations array. By doing so, we avoid redundant documents and ensure that late or retried streaming events don’t generate duplicates.

Scalability and Fault-Tolerance: The data model is designed to support high ingestion rates during peak travel periods. By isolating frequently updated data (violations) from static metadata (vehicles and cameras), we minimize write amplification and contention. The flat structure of the violation documents, along with upserts rather than inserts, reduces the chance of write conflicts and simplifies downstream compaction or deduplication. 
This model also supports low-latency lookups: frequent queries, such as finding all violations by a given car plate or date, are served efficiently using secondary indexes.

Trade-off:
Embedding multiple violations inside a single document per vehicle per day improves the write and query performance by reducing the document count and allowing bulk updates. This also aligns well with the expected access pattern: viewing all violations incurred by a vehicle during a particular trip or day.

However, embedding partial camera data in violation records leads to data duplication, but this trade-off avoids lookup latency and potential consistency issues that might arise from fetching data from the Camera colleciton at query time.

## <span style="color:#0b486b">Part 2: Streaming Application</span>

### Task 2.1 Data Stream Processing

Import required libraries

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, lit, from_json, to_timestamp, unix_timestamp, max, round
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import uuid

Set hostip

In [2]:
topic_name = 'events'
hostip = "172.23.224.1"

Connects to the Kafka broker and subscibes to the topic

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('Speed Violation Detector')
    .getOrCreate()
)

topic_stream_df = (
    spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', f'{hostip}:9092')
    .option('subscribe', topic_name)
    .option('startingOffsets', 'earliest') # for debugging
    .load()
)

Write vehicle and camera data into database

In [4]:
camera_df = spark.read.csv('data/camera.csv', header=True)
vehicle_df = spark.read.csv('data/vehicle.csv', header=True)
vehicle_df = vehicle_df.withColumn("registration_date", to_timestamp("registration_date"))

# Only get the latest registered owner of the car
latest_dates_df = vehicle_df.groupBy('car_plate').agg(
    max('registration_date').alias('latest_reg_date')
)

v1 = vehicle_df.alias("v1")
v2 = latest_dates_df.alias("v2")

latest_vehicle_df = v1.join(
    v2,
    (v1.car_plate == v2.car_plate) &
    (v1.registration_date == v2.latest_reg_date)
).select(v1['car_plate'], v1['owner_name'], v1['owner_addr'], v1['vehicle_type'], v1['registration_date'])

In [5]:
client = MongoClient(host=hostip, port=27017)
db = client['fit3182_db']

# Create violations collection and create indexes
db.violations.create_index(
    [('car_plate', 1), ('violation_date', 1)],
    unique=True
)

# Create cameras collection and create indexes
camera_pd_df = camera_df.toPandas()
camera_records = camera_pd_df.to_dict(orient='records')
db.cameras.create_index("camera_id", unique=True)

try:
    db.cameras.insert_many(camera_records, ordered=False)
    print("Inserted camera records.")
except Exception as e:
    print("Camera insert failed:", e)

# Create vehicles collection and create indexes
vehicle_pd_df = latest_vehicle_df.toPandas()
vehicle_records = vehicle_pd_df.to_dict(orient='records')
db.vehicles.create_index("car_plate", unique=True)

try:
    db.vehicles.insert_many(vehicle_records, ordered=False)
    print("Inserted vehicle records.")
except Exception as e:
    print("Vehicle insert failed:", e)

Inserted camera records.


/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


Inserted vehicle records.


Parse events from Kafka stream

In [6]:
# Define schema
camera_event_schema = StructType([
    StructField('producer_id', StringType()),
    StructField('car_plate', StringType()),
    StructField('camera_id', IntegerType()),
    StructField('timestamp', StringType()),
    StructField('speed_reading', DoubleType())
])

# Parse and convert format
json_df = topic_stream_df.select(col('value').cast('string'))
output_stream_df = json_df.select(
    from_json(col('value'), camera_event_schema).alias('data')
).select("data.*")

output_stream_df = output_stream_df.withColumn(
    'timestamp', to_timestamp('timestamp')
)

Join with camera data to get speed limit

In [7]:
camera_df = camera_df.withColumn(
  'camera_id', col('camera_id').cast('int')  
).withColumn(
  'position', col('position').cast('float')  
).withColumn('speed_limit', col('speed_limit').cast('float'))

camera_pos_dict = {
    row['camera_id']: row['position']
    for row in camera_df.select('camera_id', 'position').collect()
}

# Get distance between cameras
DIST_AB = camera_pos_dict[2] - camera_pos_dict[1]
DIST_BC = camera_pos_dict[3] - camera_pos_dict[2]

# Join stream with camera data
camera_speed_limit_df = camera_df.select('camera_id', 'speed_limit')
events_df = output_stream_df.join(camera_speed_limit_df, on='camera_id')

Detect instantaneous speed violations

In [8]:
instantaneous_violation_df = events_df.filter(
    col('speed_reading') > col('speed_limit')
).withColumn(
    'violation_type', lit('instantaneous')
).withColumn(
    'speed', round(col('speed_reading'), 1)
).select(
    'car_plate',
    'violation_type',
    'camera_id',
    'timestamp',
    'speed',
    'speed_limit'
)

Detect average speed violations

In [9]:
# Apply watermark
events_with_watermark = events_df.withWatermark(
    'timestamp', '10 minutes'
)

# Split streams by producer
a_df = events_with_watermark.filter(
    col('producer_id') == 'A'
).select(
    col('car_plate'), col('camera_id'), col('timestamp').alias('t_a'), col('speed_limit')
).alias('a')

b_df = events_with_watermark.filter(
    col('producer_id') == 'B'
).select(
    col('car_plate'), col('camera_id'), col('timestamp').alias('t_b'), col('speed_limit')
).alias('b')

c_df = events_with_watermark.filter(
    col('producer_id') == 'C'
).select(
    col('car_plate'), col('camera_id'), col('timestamp').alias('t_c'), col('speed_limit')
).alias('c')

# Detect average speed violations between camera A and B
ab_df = a_df.join(
    b_df,
    (a_df.car_plate == b_df.car_plate) &
    (b_df.t_b > a_df.t_a) &
    (b_df.t_b <= a_df.t_a + expr('interval 40 seconds'))
).select(
    col('a.car_plate'),
    col('b.camera_id').alias('camera_id'),
    col('t_a').alias('start_time'),
    col('t_b').alias('timestamp'),
    col('b.speed_limit')
).withColumn(
    'average_speed_raw', 
    expr(f"{DIST_AB} / ((unix_timestamp(timestamp) - unix_timestamp(start_time)) / 3600)")
).withColumn(
    'speed', round(col('average_speed_raw'), 1)
).filter(
    col('speed') > col('b.speed_limit')
).withColumn(
    'violation_type', lit('average_speed')
)

# Detect average speed violations between camera B and C
bc_df = b_df.join(
    c_df,
    (b_df.car_plate == c_df.car_plate) &
    (c_df.t_c > b_df.t_b) &
    (c_df.t_c <= b_df.t_b + expr('interval 40 seconds'))
).select(
    col('b.car_plate'), 
    col('c.camera_id').alias('camera_id'),
    col('t_b').alias('start_time'),
    col('t_c').alias('timestamp'),
    col('c.speed_limit')
).withColumn(
    'average_speed_raw', 
    expr(f"{DIST_BC} / ((unix_timestamp(timestamp) - unix_timestamp(start_time)) / 3600)")
).withColumn(
    'speed', round(col('average_speed_raw'), 1)
).filter(
    col('speed') > col('c.speed_limit')
).withColumn(
    'violation_type', lit('average_speed')
)

# Merge all violations
avg_speed_violations_df = ab_df.unionByName(bc_df)
avg_speed_violations_df = avg_speed_violations_df.select(
    'car_plate',
    'violation_type',
    'camera_id',
    'timestamp',
    'speed',
    'speed_limit'
)

Merge instantaneous and average speed violations

In [10]:
all_violations_df = avg_speed_violations_df.unionByName(
    instantaneous_violation_df
).withColumn('speed', round(col('speed'), 1).cast(DoubleType()))

Detect all non-violations

In [11]:
instantaneous_non_violation_df = events_df.filter(
    col('speed_reading') <= col('speed_limit')
).withColumn(
    'speed', round(col('speed_reading'), 1)
).select(
    'car_plate',
    'camera_id',
    'timestamp',
    'speed',
    'speed_limit'
)

ab_non_df = a_df.join(
    b_df,
    (a_df.car_plate == b_df.car_plate) &
    (b_df.t_b > a_df.t_a + expr('interval 40 seconds'))
).select(
    col('a.car_plate'),
    col('b.camera_id').alias('camera_id'),
    col('t_a').alias('start_time'),
    col('t_b').alias('timestamp'),
    col('b.speed_limit')
).withColumn(
    'average_speed_raw', 
    expr(f"{DIST_AB} / ((unix_timestamp(timestamp) - unix_timestamp(start_time)) / 3600)")
).withColumn(
    'speed', round(col('average_speed_raw'), 1)
).filter(
    col('speed') <= col('b.speed_limit')
)

bc_non_df = b_df.join(
    c_df,
    (b_df.car_plate == c_df.car_plate) &
    (c_df.t_c > b_df.t_b + expr('interval 40 seconds'))
).select(
    col('b.car_plate'), 
    col('c.camera_id').alias('camera_id'),
    col('t_b').alias('start_time'),
    col('t_c').alias('timestamp'),
    col('c.speed_limit')
).withColumn(
    'average_speed_raw', 
    expr(f"{DIST_BC} / ((unix_timestamp(timestamp) - unix_timestamp(start_time)) / 3600)")
).withColumn(
    'speed', round(col('average_speed_raw'), 1)
).filter(
    col('speed') <= col('c.speed_limit')
)

# Merge all non-violations
avg_non_violations_df = ab_non_df.unionByName(bc_non_df)
avg_non_violations_df = avg_non_violations_df.select(
    'car_plate',
    'camera_id',
    'timestamp',
    'speed',
    'speed_limit'
)
all_non_violations_df = instantaneous_non_violation_df.unionByName(
    avg_non_violations_df
)

Write violations to MongoDB

In [12]:
def write_batch(batch_df, epoch_id):
    print(f"===> Processing batch {epoch_id}")
    # Skip processing if the batch is empty
    if batch_df.rdd.isEmpty():
        return
    
    client = MongoClient(host=hostip, port=27017)
    db = client['fit3182_db']
    collection = db['violations']
    
    operations = []
    
    for row in batch_df.collect():
        try:
            car_plate = row['car_plate']
            timestamp = row['timestamp']
            violation_date = timestamp.date().isoformat()
            
            # Define the violation record to store
            violation_entry = {
                'violation_type': row['violation_type'],
                'camera_id': row['camera_id'],
                'timestamp': row['timestamp'],
                'speed': row['speed']
            }
            
            filter_query = {'car_plate': car_plate, 'violation_date': violation_date}
            update_query = {
                '$setOnInsert': {
                    'violation_id': str(uuid.uuid4()),
                    'car_plate': car_plate,
                    'violation_date': violation_date,
                },
                '$addToSet': {
                    'all_violations': violation_entry
                }
            }
            
            operations.append(UpdateOne(filter_query, update_query, upsert=True))
            
        except Exception as e:
            print(f"Failed to process row: {e}")
            
    if operations:
        print(f"Preparing to write {len(operations)} operations")
        try:
            collection.bulk_write(operations, ordered=False)
            print(f"Wrote {len(operations)} operations.")
        except Exception as ex:
            print(f"Bulk write failed: {ex}")
            
    client.close()

Define stream writer

In [13]:
db_writer = (
    all_violations_df
    .writeStream
    .outputMode('append')
    .foreachBatch(write_batch)
)

writer = db_writer

Log all non-violations to console

In [14]:
console_logger = (
    all_non_violations_df
    .writeStream
    .outputMode('append')
    .format('console')
)

Run the streaming query

In [ ]:
try:
    query = writer.start()
    console_query = console_logger.start()
    query.awaitTermination()
    console_query.awaitTermination()
except KeyboardInterrupt:
    print('Interrupted by CTRL-C. Stopped query')
except StreamingQueryException as exc:
    print(exc)
finally:
    query.stop()
    console_query.stop()

===> Processing batch 0
Preparing to write 151 operations
Wrote 151 operations.
===> Processing batch 1
Preparing to write 537 operations
Wrote 537 operations.
===> Processing batch 2
Preparing to write 286 operations
Wrote 286 operations.
===> Processing batch 3
Preparing to write 234 operations
Wrote 234 operations.
===> Processing batch 4
Preparing to write 284 operations
Wrote 284 operations.
===> Processing batch 5
